# 18 — Corrective RAG

> **Tier 4 | Agentic & Self-Improving**

## What is Corrective RAG?

Standard RAG blindly feeds whatever it retrieves into the LLM — even when chunks
are irrelevant. **Corrective RAG (CRAG)** wraps retrieval with a self-evaluation loop:

1. **Retrieve** top-K chunks as usual
2. **Grade** every chunk: ask the LLM to score relevance 0–10
3. **Route** based on the aggregate confidence score:

| Route | Condition | Action |
|-------|-----------|--------|
| `USE_AS_IS` | avg ≥ 0.7 | Feed retrieved chunks directly to the LLM |
| `REFINE` | 0.4 ≤ avg < 0.7 | Keep high-confidence chunks; refine query for the rest and re-retrieve |
| `FALLBACK` | avg < 0.4 | Discard all; generate a broader query and re-retrieve |

4. **Generate** the final answer from the corrected context

## PDF Framework: pdfplumber

This notebook introduces **pdfplumber** (notebooks 01–17 used `pypdf`).
pdfplumber provides richer per-page metadata that enriches chunk payloads:

| Feature | `pypdf` | `pdfplumber` |
|---------|---------|-------------|
| Text extraction | ✓ | ✓ |
| Word bounding boxes | ✗ | ✓ `page.extract_words()` |
| Page density (words/area) | ✗ | ✓ derived metric |
| Table detection | ✗ | ✓ `page.extract_tables()` |
| Crop / region filter | ✗ | ✓ |

We store `bbox_density` and `has_tables` on every chunk so the grader has
structural signals beyond text similarity.

## Flow Diagram

```mermaid
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pdfplumber"]
        PDF(["📄 climate.pdf"])
        PDF --> PB["pdfplumber.open()\nper-page extraction"]
        PB --> PMETA["Page stats:\nbbox_density, has_tables"]
        PMETA --> SPLIT["Fixed-size chunks\n~500 chars"]
        SPLIT --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\ncorrective_rag_18")]
    end

    subgraph RETRIEVE ["🔍  RETRIEVE"]
        Q(["❓ Query"])
        Q --> QEMB["embed(query)"]
        QEMB --> SEARCH["Top-K search"]
        QDRANT --> SEARCH
        SEARCH --> CHUNKS(["Retrieved chunks"])
    end

    subgraph GRADE ["🟡  GRADE & ROUTE"]
        CHUNKS --> GRADER["LLM Grader\n0–10 per chunk"]
        GRADER --> AVG["avg_score"]
        AVG --> ROUTE{{"Route"}}
        ROUTE -->|"avg ≥ 0.7"| USE["✅ USE_AS_IS"]
        ROUTE -->|"0.4 ≤ avg < 0.7"| REFINE["🔄 REFINE\nrefine query + re-retrieve"]
        ROUTE -->|"avg < 0.4"| FALLBACK["⚠️ FALLBACK\nbroader query + re-retrieve"]
    end

    subgraph GEN ["🟠  GENERATION"]
        USE --> CTX["Corrected context"]
        REFINE --> CTX
        FALLBACK --> CTX
        CTX --> LLM["Strands Agent\nClaude Sonnet 4.6"]
        LLM --> ANS(["✅ Answer + route label"])
    end

    style INDEX    fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style RETRIEVE fill:#f0fdf4,stroke:#16a34a,color:#14532d
    style GRADE    fill:#fef9c3,stroke:#ca8a04,color:#713f12
    style GEN      fill:#ffedd5,stroke:#f97316,color:#7c2d12
```


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pdfplumber"]
        PDF(["📄 climate.pdf"])
        PDF --> PB["pdfplumber.open()\nper-page extraction"]
        PB --> PMETA["Page stats:\nbbox_density, has_tables"]
        PMETA --> SPLIT["Fixed-size chunks\n~500 chars"]
        SPLIT --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\ncorrective_rag_18")]
    end

    subgraph RETRIEVE ["🔍  RETRIEVE"]
        Q(["❓ Query"])
        Q --> QEMB["embed(query)"]
        QEMB --> SEARCH["Top-K search"]
        QDRANT --> SEARCH
        SEARCH --> CHUNKS(["Retrieved chunks"])
    end

    subgraph GRADE ["🟡  GRADE & ROUTE"]
        CHUNKS --> GRADER["LLM Grader\n0–10 per chunk"]
        GRADER --> AVG["avg_score"]
        AVG --> ROUTE{{"Route"}}
        ROUTE -->|"avg ≥ 0.7"| USE["✅ USE_AS_IS"]
        ROUTE -->|"0.4 ≤ avg < 0.7"| REFINE["🔄 REFINE\nrefine query + re-retrieve"]
        ROUTE -->|"avg < 0.4"| FALLBACK["⚠️ FALLBACK\nbroader query + re-retrieve"]
    end

    subgraph GEN ["🟠  GENERATION"]
        USE --> CTX["Corrected context"]
        REFINE --> CTX
        FALLBACK --> CTX
        CTX --> LLM["Strands Agent\nClaude Sonnet 4.6"]
        LLM --> ANS(["✅ Answer + route label"])
    end

    style INDEX    fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style RETRIEVE fill:#f0fdf4,stroke:#16a34a,color:#14532d
    style GRADE    fill:#fef9c3,stroke:#ca8a04,color:#713f12
    style GEN      fill:#ffedd5,stroke:#f97316,color:#7c2d12
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "strands-agents", "pdfplumber", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, re
from typing import List, Dict, Optional, Tuple
from dotenv import load_dotenv

import boto3, pdfplumber
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")


## Step 2 — Configuration

In [ ]:
# AWS / Bedrock
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

# Vector DB
QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

# Collection
COLLECTION_NAME = "corrective_rag_18"
EMBEDDING_DIM   = 1024
TOP_K           = 6

# Chunking
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

# Corrective RAG thresholds
USE_AS_IS_THRESH = 0.7   # avg chunk score ≥ this → use as-is
FALLBACK_THRESH  = 0.4   # avg chunk score < this → full fallback

# PDF
PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection      : {COLLECTION_NAME}")
print(f"PDF             : {PDF_PATH}")
print(f"PDF exists      : {os.path.exists(PDF_PATH)}")
print(f"USE_AS_IS_THRESH: {USE_AS_IS_THRESH}")
print(f"FALLBACK_THRESH : {FALLBACK_THRESH}")


## Step 3 — Vector Store

In [ ]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name     = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k, with_payload=True)
            return [{'text': p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        print(f'Deleted "{self.name}"')

print("VectorStore defined.")


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str, system: str = "You are a helpful assistant.") -> str:
    return str(Agent(model=_model, system_prompt=system)(prompt))

test_emb = embed_text("corrective rag test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Strands BedrockModel ready.")


## Step 5 — PDF Loading with pdfplumber

`pdfplumber` gives us per-page word bounding boxes, which we use to compute
`bbox_density` (words per unit area) — a proxy for information density.
Pages with tables also get a `has_tables=True` flag.
Both fields are stored in the chunk payload alongside the usual metadata.


In [ ]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)


def load_pdf_pdfplumber(path: str) -> Tuple[List[Dict], Dict]:
    """
    Returns:
      chunks  — list of {text, page_num, bbox_density, has_tables}
      stats   — extraction summary
    """
    chunks = []
    page_stats = []

    with pdfplumber.open(path) as pdf:
        n_pages = len(pdf.pages)
        for page in pdf.pages:
            page_num   = page.page_number          # 1-indexed
            text       = page.extract_text() or ''

            # Bounding-box density: words / page area (normalised)
            words      = page.extract_words()
            area       = (page.width * page.height) or 1
            bbox_dens  = round(len(words) / area * 1000, 3)  # words per 1k px²

            # Table detection
            tables     = page.extract_tables()
            has_tables = len(tables) > 0

            page_chunks = recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP)
            for chunk in page_chunks:
                chunks.append({
                    'text'        : chunk,
                    'page_num'    : page_num,
                    'bbox_density': bbox_dens,
                    'has_tables'  : has_tables,
                    'char_count'  : len(chunk),
                    'word_count'  : len(chunk.split()),
                })
            page_stats.append({
                'page': page_num, 'words': len(words),
                'bbox_density': bbox_dens, 'has_tables': has_tables,
                'chunks': len(page_chunks)
            })

    stats = {
        'n_pages'       : n_pages,
        'n_chunks'      : len(chunks),
        'avg_chunk_len' : sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
        'pages_with_tables': sum(1 for p in page_stats if p['has_tables']),
        'avg_bbox_density' : round(sum(p['bbox_density'] for p in page_stats) / n_pages, 3),
    }
    return chunks, stats, page_stats


t0 = time.time()
chunks, stats, page_stats = load_pdf_pdfplumber(PDF_PATH)
elapsed = time.time() - t0

print(f"pdfplumber extraction: {elapsed*1000:.0f}ms")
print(f"Pages            : {stats['n_pages']}")
print(f"Chunks           : {stats['n_chunks']}")
print(f"Avg chunk length : {stats['avg_chunk_len']} chars")
print(f"Pages with tables: {stats['pages_with_tables']}")
print(f"Avg bbox_density : {stats['avg_bbox_density']} words/1kpx²")
print()
print(f"{'Page':>5}  {'Words':>6}  {'Density':>9}  {'Tables':>7}  {'Chunks':>7}")
print("-" * 44)
for p in page_stats:
    print(f"{p['page']:>5}  {p['words']:>6}  {p['bbox_density']:>9}  "
          f"{'yes' if p['has_tables'] else 'no':>7}  {p['chunks']:>7}")


## Step 6 — Connect & Index

In [ ]:
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL, qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL, region=AWS_REGION)
print(f"Backend: {vs._backend}")

vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
texts = [c['text'] for c in chunks]
embs  = embed_batch(texts, label='[index]')

docs = [
    {
        'text'     : chunks[i]['text'],
        'embedding': embs[i],
        'metadata' : {
            'chunk_idx'   : i,
            'page_num'    : chunks[i]['page_num'],
            'bbox_density': chunks[i]['bbox_density'],
            'has_tables'  : chunks[i]['has_tables'],
            'char_count'  : chunks[i]['char_count'],
            'word_count'  : chunks[i]['word_count'],
            'source'      : 'climate.pdf',
            'pdf_lib'     : 'pdfplumber',
        }
    }
    for i in range(len(chunks))
]
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors in {time.time()-t0:.1f}s")


## Step 7 — Chunk Grader

The grader asks the LLM to score each retrieved chunk 0–10 for relevance
to the question. We parse the integer from the response and normalise to 0–1.

The grader prompt is compact (1-shot instruction + question + chunk ≤ 200 chars)
so each grading call is fast.


In [ ]:
GRADE_SYSTEM = (
    "You are a relevance grader. Given a question and a text chunk, "
    "output ONLY an integer 0-10 representing how relevant the chunk is "
    "to answering the question. 0 = completely irrelevant, 10 = perfectly answers it. "
    "Output the number alone with no explanation."
)

def grade_chunk(question: str, chunk_text: str) -> float:
    prompt = (
        f"Question: {question}\n\n"
        f"Chunk (first 300 chars): {chunk_text[:300]}\n\n"
        "Relevance score (0-10):"
    )
    raw = llm(prompt, system=GRADE_SYSTEM).strip()
    # Extract first integer found
    m = re.search(r'\b(\d+)\b', raw)
    if m:
        score = min(10, max(0, int(m.group(1))))
        return score / 10.0
    return 0.5  # neutral fallback if parsing fails

def grade_chunks(question: str, hits: List[Dict]) -> List[Dict]:
    graded = []
    for h in hits:
        score = grade_chunk(question, h['text'])
        graded.append({**h, 'grade': score})
        print(f"  chunk[p{h['metadata'].get('page_num','?')}] "
              f"vec_score={h['score']:.3f}  grade={score:.1f}  "
              f"'{h['text'][:60]}...'")
    return graded

# Unit test the grader
_test_q   = "What is the role of satellites in weather forecasting?"
_test_hit = {'text': 'Satellites provide real-time atmospheric data for weather models.',
             'metadata': {'page_num': 5}, 'score': 0.85}
_test_irr = {'text': 'The capital of France is Paris.',
             'metadata': {'page_num': 1}, 'score': 0.20}

print("Grader tests:")
print(f"  relevant  : {grade_chunk(_test_q, _test_hit['text']):.1f}  (expect ≥ 0.6)")
print(f"  irrelevant: {grade_chunk(_test_q, _test_irr['text']):.1f}  (expect ≤ 0.4)")
print("Grader OK")


## Step 8 — Router & Query Rewriter

`route()` maps the average grade to one of three labels.

`rewrite_query()` asks the LLM to produce a refined or broader version of the
original question — used in the REFINE and FALLBACK paths respectively.


In [ ]:
ROUTE_USE_AS_IS = "USE_AS_IS"
ROUTE_REFINE    = "REFINE"
ROUTE_FALLBACK  = "FALLBACK"

def route(graded_hits: List[Dict]) -> Tuple[str, float]:
    if not graded_hits:
        return ROUTE_FALLBACK, 0.0
    avg = sum(h['grade'] for h in graded_hits) / len(graded_hits)
    if avg >= USE_AS_IS_THRESH:
        return ROUTE_USE_AS_IS, avg
    elif avg >= FALLBACK_THRESH:
        return ROUTE_REFINE, avg
    else:
        return ROUTE_FALLBACK, avg

REWRITE_SYSTEM = (
    "You are a query rewriter. Rewrite the given question to improve retrieval. "
    "Output ONLY the rewritten question, nothing else."
)

def rewrite_query(question: str, mode: str) -> str:
    if mode == ROUTE_REFINE:
        instruction = (
            "Rephrase this question to be more specific and targeted, "
            "focusing on the exact information needed:"
        )
    else:  # FALLBACK
        instruction = (
            "Rephrase this question to be broader and more general "
            "so it can retrieve relevant background information:"
        )
    prompt = f"{instruction}\n\nOriginal: {question}\n\nRewritten:"
    return llm(prompt, system=REWRITE_SYSTEM).strip()

# Quick test
_rq = "What is weather forecasting?"
print(f"REFINE   rewrite: {rewrite_query(_rq, ROUTE_REFINE)}")
print(f"FALLBACK rewrite: {rewrite_query(_rq, ROUTE_FALLBACK)}")


## Step 9 — Full Corrective RAG Pipeline

`corrective_rag()` wires together all steps:

1. Retrieve top-K chunks
2. Grade every chunk
3. Route based on avg grade
4. If REFINE: keep high-grade chunks, rewrite query, re-retrieve for the rest
5. If FALLBACK: rewrite query broadly, re-retrieve from scratch
6. Generate the final answer from the corrected context


In [ ]:
ANSWER_SYSTEM = (
    "You are a precise Q&A assistant. Answer only from the provided context. "
    "If the answer is not present say 'Not found in context.'"
)

def generate_answer(question: str, context_chunks: List[str]) -> str:
    context = '\n\n'.join(f'[Passage {i+1}]\n{c}' for i, c in enumerate(context_chunks))
    prompt  = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    return llm(prompt, system=ANSWER_SYSTEM)


def corrective_rag(question: str, top_k: int = TOP_K, verbose: bool = True) -> Dict:
    t0 = time.time()

    # ── 1. Initial retrieval ──
    qvec = embed_text(question)
    hits = vs.search(qvec, top_k=top_k)

    if verbose:
        print(f"\nQ: {question}")
        print(f"Initial retrieval: {len(hits)} chunks")

    # ── 2. Grade ──
    if verbose: print("Grading chunks...")
    graded = grade_chunks(question, hits)
    decision, avg_grade = route(graded)

    if verbose:
        print(f"Avg grade: {avg_grade:.2f}  →  Route: {decision}")

    # ── 3. Route ──
    final_chunks: List[str] = []

    if decision == ROUTE_USE_AS_IS:
        final_chunks = [h['text'] for h in graded]

    elif decision == ROUTE_REFINE:
        # Keep chunks above median grade; re-retrieve for the rest
        median_grade = sorted(h['grade'] for h in graded)[len(graded) // 2]
        good  = [h['text'] for h in graded if h['grade'] >= median_grade]
        n_bad = len(graded) - len(good)
        if verbose:
            print(f"  Keeping {len(good)} good chunks, re-retrieving {n_bad} slots")
        rewritten = rewrite_query(question, ROUTE_REFINE)
        if verbose: print(f"  Refined query: {rewritten}")
        extra_hits = vs.search(embed_text(rewritten), top_k=n_bad + 2)
        seen = {h['text'][:80] for h in graded}
        fresh = [h['text'] for h in extra_hits if h['text'][:80] not in seen][:n_bad]
        final_chunks = good + fresh

    else:  # FALLBACK
        rewritten = rewrite_query(question, ROUTE_FALLBACK)
        if verbose: print(f"  Fallback query: {rewritten}")
        fallback_hits = vs.search(embed_text(rewritten), top_k=top_k)
        final_chunks  = [h['text'] for h in fallback_hits]

    # ── 4. Generate ──
    answer  = generate_answer(question, final_chunks[:top_k])
    latency = (time.time() - t0) * 1000

    if verbose:
        print(f"\nAnswer ({latency:.0f}ms):\n{answer}")
        print("-" * 70)

    return {
        'question'    : question,
        'route'       : decision,
        'avg_grade'   : round(avg_grade, 3),
        'n_final_ctx' : len(final_chunks),
        'answer'      : answer,
        'latency_ms'  : latency,
    }


# Demo — one question through the full pipeline
result = corrective_rag("What instruments are used to measure atmospheric conditions?")


## Step 10 — Multi-Question Demo (exercising all 3 routes)

In [ ]:
test_questions = [
    # Should hit USE_AS_IS — very on-topic for climate.pdf
    "What are the main methods used in weather forecasting?",
    "How do satellites contribute to weather observation?",
    # Likely REFINE — partially relevant
    "What is the historical development of meteorological instruments?",
    # Likely FALLBACK — tangential
    "What is the economic cost of inaccurate weather forecasts for agriculture?",
]

results = []
for q in test_questions:
    r = corrective_rag(q, verbose=True)
    results.append(r)

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"{'Question':<52} {'Route':<12} {'Avg grade':>9}  {'ms':>6}")
print("-" * 82)
for r in results:
    print(f"{r['question'][:51]:<52} {r['route']:<12} "
          f"{r['avg_grade']:>9.2f}  {r['latency_ms']:>6.0f}")


## Step 11 — Corrective RAG vs Simple RAG Comparison

In [ ]:
def simple_rag(question: str, top_k: int = TOP_K) -> Dict:
    t0   = time.time()
    hits = vs.search(embed_text(question), top_k=top_k)
    ans  = generate_answer(question, [h['text'] for h in hits])
    return {'answer': ans, 'latency_ms': (time.time() - t0) * 1000}

eval_questions = [
    {"q": "What methods are used in weather forecasting?",
     "kw": ["numerical", "synoptic", "forecast", "model", "observation"]},
    {"q": "How do satellites help in weather prediction?",
     "kw": ["satellite", "data", "sensor", "image", "orbit"]},
    {"q": "What is the role of Doppler radar in weather monitoring?",
     "kw": ["doppler", "radar", "wind", "velocity", "precipitation"]},
]

print(f"{'Question':<48} {'Simple KW':>9}  {'CRAG KW':>7}  {'CRAG Route':<12}")
print("-" * 84)

for eq in eval_questions:
    q   = eq['q']
    kws = eq['kw']
    n   = len(kws)

    sr = simple_rag(q)
    cr = corrective_rag(q, verbose=False)

    s_hits = sum(1 for kw in kws if kw in sr['answer'].lower())
    c_hits = sum(1 for kw in kws if kw in cr['answer'].lower())

    print(f"{q[:47]:<48} {s_hits}/{n} ({s_hits/n*100:.0f}%)  "
          f"{c_hits}/{n} ({c_hits/n*100:.0f}%)  {cr['route']:<12}")

print()
print("Note: CRAG adds grading latency per chunk but routes low-confidence")
print("      retrievals to refined queries — higher accuracy on weak-match Qs.")


## Step 12 — Summary

| Component | Implementation |
|-----------|---------------|
| PDF loading | **pdfplumber** — per-page bounding boxes, density, table detection |
| Extra metadata | `bbox_density` (words/1kpx²), `has_tables` stored in Qdrant payload |
| Chunking | Native `recursive_split()` — 500 chars, 50 overlap |
| Embeddings | Amazon Bedrock Titan V2 (1024-dim) |
| Grader | Strands Agent + Claude Sonnet 4.6 → integer 0–10 per chunk |
| Router | avg_grade ≥ 0.7 → USE_AS_IS · 0.4–0.7 → REFINE · < 0.4 → FALLBACK |
| Query rewriter | LLM prompt — specific (REFINE) or broad (FALLBACK) |
| Vector DB | Qdrant Cloud → OpenSearch → in-memory |
| LLM (answers) | Strands Agent + Claude Sonnet 4.6 |

## pdfplumber vs pypdf — what we gained

| Capability | pypdf (nb 01–17) | pdfplumber (nb 18) |
|------------|-----------------|-------------------|
| Text extraction | ✓ | ✓ |
| Per-word bounding boxes | ✗ | ✓ `extract_words()` |
| Page density metric | ✗ | ✓ → richer chunk metadata |
| Table detection | ✗ | ✓ `extract_tables()` |
| Extraction speed | ~fast | ~2–3× slower |

### PDF Framework progression

| NB | Framework | Key advantage used |
|----|-----------|-------------------|
| 01–17 | pypdf | Simple, fast text extraction |
| **18** | **pdfplumber** | **Bounding-box density + table detection** |
| 19 | pymupdf | Speed for iterative loops |
| 20 | pdfminer.six | Fine-grained layout tree |
| 21 | unstructured | Typed elements for recursive splits |
| 22 | docling | Structured JSON/markdown |
| 23–28 | various | See notebook headers |
| 29 | All 6 | Full benchmark comparison |

### Next: **19 — Self RAG** (pymupdf)


In [ ]:
# vs.delete_collection()  # uncomment to clean up
print(f"Collection '{COLLECTION_NAME}' in {vs._backend} — {vs.count()} vectors")
print(f"PDF framework  : pdfplumber")
print(f"Chunks indexed : {stats['n_chunks']}")
print(f"Thresholds     : USE_AS_IS≥{USE_AS_IS_THRESH}  FALLBACK<{FALLBACK_THRESH}")
print("\nNotebook 18 complete. Give the go-ahead for notebook 19 (Self RAG / pymupdf).")
